## Fixed-point error propagation

Copyright 2023 Google LLC. Adapted under Apache-2.0 from Qualtran v0.7.0, commit `bcb17bb25360ed68ce3bc084645db3e929799389`.

All eight original sections/cells are retained. The numerical helpers use unsigned stored integers and the existing native fixed-point decoder. The reference fxpmath API stays outside native code. This scalar teaching asset emits no quantum circuit.


In [ ]:
import numpy as np

from examples.algorithms.fixed_point_error_analysis import (
    assert_allclose,
    get_bitsize_for_fxp_mul,
    get_bitsize_for_fxp_mul_with_integer,
    get_bitsize_for_fxp_square,
    mul_via_repeated_add,
    mul_with_int_via_repeated_add,
    square_via_repeated_add,
)


###  Appendix D4: Multiplying an integer to a real number
Goal: Given quantum registers A and B with real number $\kappa$ ($0 \leq \kappa \lt 1$) and a $d_{B}$-bit integer $\lambda$, our goal is to compute an approximation $\widetilde{\gamma}$ of $\gamma = \kappa * \lambda$ s.t. $|\widetilde{\gamma} - \gamma| < \epsilon$

$$
    |\kappa\rangle_{A} |\lambda\rangle_{B} |0\rangle_{\text{out}} \rightarrow |\kappa\rangle_{A} |\lambda\rangle_{B} |\widetilde{\gamma}\rangle_{\text{out}}
$$


We analyze the error due to fixed point arithmetic for algorithm described in Appendix D4 of https://arxiv.org/abs/2007.07391

In [ ]:
# Multiplying a real numbers with an integer_width-bit integer.


def test_multiplication_with_integer_for_eps(
    eps: float, integer_width: int, real_width: int
) -> None:
    rng = np.random.default_rng(int(eps * integer_width * 1e9))
    try:
        for _ in range(100):
            (a,) = rng.random(1)
            b = int(rng.integers(0, 1 << integer_width))
            res = mul_with_int_via_repeated_add(a, b, real_width, integer_width)
            assert_allclose(res, a * b, eps)
        print(f"Success! {eps=}, d_A={real_width}, d_B={integer_width}")
    except AssertionError:
        print(f"Failed! {eps=}, d_A={real_width}, d_B={integer_width}")


for eps in [1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-9]:
    for integer_width in [5, 8, 11]:
        real_width = get_bitsize_for_fxp_mul_with_integer(eps, integer_width)
        test_multiplication_with_integer_for_eps(eps, integer_width, real_width)


###  Appendix D5: Multiplying two different real numbers
Goal: Given quantum registers A and B with real numbers $\kappa$ ($0 \leq \kappa \lt 1$) and $\lambda$ ($0 \leq \lambda \lt 1$), our goal is to compute an approximation $\widetilde{\gamma}$ of $\gamma = \kappa * \lambda$ s.t. $|\widetilde{\gamma} - \gamma| < \epsilon$

$$
    |\kappa\rangle_{A} |\lambda\rangle_{B} |0\rangle_{\text{out}} \rightarrow |\kappa\rangle_{A} |\lambda\rangle_{B} |\widetilde{\gamma}\rangle_{\text{out}}
$$


We analyze the error due to fixed point arithmetic for algorithm described in Appendix D5 of https://arxiv.org/abs/2007.07391

In [ ]:
# Multiplying two real numbers


def test_multiplication_for_eps(eps: float, d: int) -> None:
    rng = np.random.default_rng(int(eps * 1e9))
    try:
        for _ in range(100):
            a, b = rng.random(2)
            res = mul_via_repeated_add(a, b, d)
            assert_allclose(res, a * b, eps)
        print(f"Success! {d=}, {eps=}")
    except AssertionError:
        print(f"Failed! {d=}, {eps=}")


for eps in [1e-3, 1e-4, 1e-5, 1e-6, 1e-7]:
    d = get_bitsize_for_fxp_mul(eps)
    test_multiplication_for_eps(
        eps, d - 1
    )  # Works for d-1 as well, bounds in the paper are probably loose?


###  Appendix D6: Squaring a real number
Goal: Given a quantum register A with a real number $\kappa$ ($0 \leq \kappa \lt 1$), our goal is to compute an approximation $\widetilde{\gamma}$ of $\gamma = \kappa^2$ s.t. $|\widetilde{\gamma} - \gamma| < \epsilon$

$$
    |\kappa\rangle_{A} |0\rangle_{\text{out}} \rightarrow |\kappa\rangle_{A} |\widetilde{\gamma}\rangle_{\text{out}}
$$


We analyze the error due to fixed point arithmetic for algorithm described in Appendix D6 of https://arxiv.org/abs/2007.07391

In [ ]:
# Squaring a real number


def test_square_for_eps(eps: float, d: int) -> None:
    rng = np.random.default_rng(int(eps * 1e9))
    try:
        for _ in range(100):
            (a,) = rng.random(1)
            res = square_via_repeated_add(a, d)
            assert_allclose(res, a**2, eps)
        print(f"Success! {d=}, {eps=}")
    except AssertionError:
        print(f"Failed! {d=}, {eps=}")


for eps in [1e-3, 1e-4, 1e-5, 1e-6, 1e-7]:
    d = get_bitsize_for_fxp_square(
        eps
    )  # Need a +1 to make it work for smaller `eps`. Bound in the paper is too tight?
    test_square_for_eps(eps, d)
# Original D36 samples fail for eps=1e-3, 1e-4 and 1e-5.
# Preserve these diagnostics instead of adding a precision bit.
